# LeetCode 4083 — Minimum Deletions to Make Array Sharp bằng Python

Link đề: https://leetcode.com/problems/minimum-deletions-to-make-array-sharp/ · Độ khó: **Easy** · Chủ đề: Array, Bit Manipulation, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minDeletions(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ lấy từ đề gốc (Weekly Contest 524, câu 1); mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng hai lời giải độc lập: **duyệt mọi tập con** (cho mảng ngắn) và **quy hoạch động `O(n²)`** (cho mảng dài).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: được **xóa tùy ý** một số phần tử (các phần tử còn lại **giữ nguyên thứ tự**); mảng còn lại gọi là **sắc (sharp)** nếu **XOR của mọi cặp phần tử kề nhau khác 0**; tìm **số lần xóa ít nhất** để `nums` thành mảng sắc.
2. Biết `a XOR b == 0` **khi và chỉ khi `a == b`**, nên điều kiện "sắc" đơn giản là **không có hai phần tử kề nhau bằng nhau**.
3. Đi từ **duyệt mọi tập con (brute force)** → **quy hoạch động `O(n²)`** → **tham lam bằng ngăn xếp** → **đếm số cặp kề bằng nhau** → **đếm các đoạn bằng nhau (`groupby`)**, và hiểu vì sao đáp án chính là **số cặp kề bằng nhau** = `n − số đoạn`.
4. Làm quen toán tử **XOR (`^`)**, **mảng con liên tiếp ("run") các giá trị bằng nhau**, `itertools.pairwise`, `itertools.groupby`.
5. Tránh các lỗi kinh điển: xóa **cả hai** phần tử của một cặp bằng nhau, đếm số giá trị **trùng** (không cần kề nhau), so sánh vòng tròn `nums[-1]`, và hiểu ngược điều kiện (`x ^ y != 0` đếm các cặp **khác** nhau).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`.

Bạn có thể **xóa** bất kỳ số phần tử nào khỏi `nums`; các phần tử còn lại **giữ nguyên thứ tự ban đầu**. Mảng kết quả `a` được gọi là **sắc (sharp)** nếu `a[i] XOR a[i + 1] != 0` với mọi chỉ số `i` thỏa `0 <= i < a.length − 1`. Mảng có **nhiều nhất một phần tử** luôn là mảng sắc.

Trả về **số lần xóa nhỏ nhất** để `nums` trở thành mảng sắc.

(XOR bit là phép toán so sánh từng bit tương ứng: bit kết quả là `1` nếu hai bit khác nhau, và `0` nếu giống nhau.)

**Ví dụ 1**

- Input: `nums = [1,1,2,2,2,3]`
- Output: `3`
- Giải thích: các cặp chỉ số `(0,1)`, `(2,3)` và `(3,4)` có XOR bằng `0`. Xóa một số `1` và hai số `2` còn lại `[1, 2, 3]`, trong đó `1 XOR 2 = 3` và `2 XOR 3 = 1` đều khác 0. Đáp án `3`.

**Ví dụ 2**

- Input: `nums = [4,4,9,4]`
- Output: `1`
- Giải thích: cặp `(0,1)` có XOR bằng `0`. Xóa một trong hai số `4` đầu tiên còn `[4, 9, 4]` có `4 XOR 9 = 13` và `9 XOR 4 = 13` đều khác 0.

**Ràng buộc:** `2 <= nums.length <= 100`; `1 <= nums[i] <= 100`.

In [1]:
from typing import List
import random
from itertools import pairwise, groupby

examples = [
    ([1, 1, 2, 2, 2, 3], 3),
    ([4, 4, 9, 4], 1),
]
for nums, expected in examples:
    print(f"nums={nums}, kỳ vọng {expected}")

nums=[1, 1, 2, 2, 2, 3], kỳ vọng 3
nums=[4, 4, 9, 4], kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Toán tử XOR `^`

`a ^ b` (XOR bit) so sánh từng bit của `a` và `b`: bit kết quả là `1` nếu hai bit **khác nhau**, là `0` nếu **giống nhau**. Quy tắc quan trọng: **`a ^ a = 0`** và `a ^ 0 = a`. Vì vậy `a ^ b = 0` khi mọi bit giống nhau, tức `a == b`.

In [2]:
for a, b in [(1, 1), (1, 2), (2, 3), (4, 9), (9, 4), (7, 7), (5, 0)]:
    print(f"{a} ^ {b} = {a ^ b}    ({a:04b} ^ {b:04b} = {a ^ b:04b})")

1 ^ 1 = 0    (0001 ^ 0001 = 0000)
1 ^ 2 = 3    (0001 ^ 0010 = 0011)
2 ^ 3 = 1    (0010 ^ 0011 = 0001)
4 ^ 9 = 13    (0100 ^ 1001 = 1101)
9 ^ 4 = 13    (1001 ^ 0100 = 1101)
7 ^ 7 = 0    (0111 ^ 0111 = 0000)
5 ^ 0 = 5    (0101 ^ 0000 = 0101)


### 3.2. `a ^ b == 0` ⇔ `a == b`

Nếu `a == b` thì mọi bit đều giống nhau nên XOR là `0`. Ngược lại nếu XOR bằng `0` thì không có bit nào khác nhau nên `a == b`. Vậy đề bài "XOR của hai phần tử kề nhau khác 0" chính là **"hai phần tử kề nhau phải khác nhau"**. Ta có thể kiểm chứng điều này trên toàn miền `1..100` của đề.

In [3]:
bad = [(a, b) for a in range(1, 101) for b in range(1, 101) if (a ^ b == 0) != (a == b)]
print("số cặp (a, b) trong 1..100 mà a ^ b == 0 khác với a == b:", len(bad))
assert bad == []

số cặp (a, b) trong 1..100 mà a ^ b == 0 khác với a == b: 0


### 3.3. Xóa phần tử nhưng giữ thứ tự (dãy con) và "đoạn" bằng nhau

Sau khi xóa, các phần tử còn lại giữ thứ tự: đó là một **dãy con** (subsequence) của `nums`. Một **đoạn bằng nhau (run)** là một khoảng **liên tiếp tối đại** các phần tử cùng giá trị. Ví dụ `[1,1,2,2,2,3]` có `3` đoạn: `[1,1]`, `[2,2,2]`, `[3]`. Hai đoạn kề nhau luôn có giá trị **khác nhau** (nếu bằng nhau thì chúng đã là một đoạn).

In [4]:
nums = [1, 1, 2, 2, 2, 3]
runs = [(value, len(list(group))) for value, group in groupby(nums)]
print(runs)               # [(1, 2), (2, 3), (3, 1)]
print("số đoạn:", len(runs), "| n - số đoạn:", len(nums) - len(runs))

[(1, 2), (2, 3), (3, 1)]
số đoạn: 3 | n - số đoạn: 3


### 3.4. `itertools.pairwise` và `groupby`

`pairwise(nums)` cho các cặp kề nhau `(nums[0], nums[1])`, `(nums[1], nums[2])`, ... (`n − 1` cặp). `groupby(nums)` gom các phần tử **liên tiếp** bằng nhau (khác `Counter`: không gom các giá trị trùng ở xa nhau).

In [5]:
nums = [4, 4, 9, 4]
print(list(pairwise(nums)))                                   # [(4, 4), (4, 9), (9, 4)]
print([k for k, _ in groupby(nums)])                          # [4, 9, 4]: ba đoạn
print(sum(x == y for x, y in pairwise(nums)))                  # 1: số cặp kề bằng nhau

[(4, 4), (4, 9), (9, 4)]
[4, 9, 4]
1


### 3.5. List dùng như ngăn xếp (stack)

Ta có thể xây dãy kết quả bằng một list `kept`: `kept.append(x)` thêm vào cuối, `kept[-1]` là phần tử cuối (phần tử được giữ gần nhất). Điều kiện `kept and kept[-1] == x` (list không rỗng và phần tử cuối bằng `x`) kiểm tra "`x` có trùng phần tử vừa giữ không".

In [6]:
kept = []
for x in [1, 1, 2]:
    if kept and kept[-1] == x:
        print(f"x={x} trùng kept[-1]={kept[-1]} -> bỏ")
    else:
        kept.append(x)
        print(f"x={x} -> giữ, kept={kept}")

x=1 -> giữ, kept=[1]
x=1 trùng kept[-1]=1 -> bỏ
x=2 -> giữ, kept=[1, 2]


### 3.6. Duyệt mọi tập con bằng bitmask

Với `n` phần tử có `2ⁿ` cách chọn giữ/xóa. Mỗi `mask` từ `0` đến `2ⁿ − 1` mã hóa một cách: bit `i` bằng `1` nghĩa là **giữ** phần tử `i`. `(mask >> i) & 1` đọc bit thứ `i`. Chỉ dùng được với `n` nhỏ (vài chục triệu thao tác khi `n ≈ 20`), nhưng rất hữu ích làm **chuẩn đối chiếu**.

In [7]:
nums = [4, 4, 9]
for mask in range(1 << len(nums)):
    kept = [nums[i] for i in range(len(nums)) if (mask >> i) & 1]
    print(f"mask={mask:03b} -> giữ {kept}")

mask=000 -> giữ []
mask=001 -> giữ [4]
mask=010 -> giữ [4]
mask=011 -> giữ [4, 4]
mask=100 -> giữ [9]
mask=101 -> giữ [4, 9]
mask=110 -> giữ [4, 9]
mask=111 -> giữ [4, 4, 9]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Làm tay ví dụ 1: `nums = [1, 1, 2, 2, 2, 3]`. Đánh dấu các cặp kề nhau:

| cặp kề `(i, i+1)` | giá trị | XOR | bằng nhau? |
|---|---|---|---|
| `(0, 1)` | `1, 1` | `0` | có |
| `(1, 2)` | `1, 2` | `3` | không |
| `(2, 3)` | `2, 2` | `0` | có |
| `(3, 4)` | `2, 2` | `0` | có |
| `(4, 5)` | `2, 3` | `1` | không |

Có `3` cặp kề bằng nhau, cần xóa để chúng không còn kề nhau bằng nhau. Cách tự nhiên: trong **mỗi đoạn các số bằng nhau**, **chỉ giữ một số**: đoạn `[1,1]` giữ 1 (xóa 1), đoạn `[2,2,2]` giữ 1 (xóa 2), đoạn `[3]` giữ nguyên. Tổng xóa `1 + 2 + 0 = 3`. Kết quả `[1, 2, 3]`.

Hai đoạn kề nhau có giá trị khác nhau nên sau khi mỗi đoạn chỉ còn một phần tử, các phần tử kề nhau đều khác nhau. Mỗi đoạn dài `k` có đúng `k − 1` cặp kề bằng nhau và ta xóa `k − 1` phần tử, vậy **số lần xóa = số cặp kề bằng nhau**.

In [8]:
nums = [1, 1, 2, 2, 2, 3]
equal_pairs = [(i, nums[i]) for i in range(len(nums) - 1) if nums[i] == nums[i + 1]]
print("các cặp kề bằng nhau (i, giá trị):", equal_pairs)
print("số cặp kề bằng nhau:", len(equal_pairs))
kept = [value for value, _ in groupby(nums)]
print("giữ một phần tử mỗi đoạn:", kept)
print("kiểm tra mảng sắc:", all(a ^ b != 0 for a, b in pairwise(kept)))

các cặp kề bằng nhau (i, giá trị): [(0, 1), (2, 2), (3, 2)]
số cặp kề bằng nhau: 3
giữ một phần tử mỗi đoạn: [1, 2, 3]
kiểm tra mảng sắc: True


## 5. Cách 1 — Brute Force / baseline: thử mọi cách giữ/xóa

**Ý tưởng.** Thử **mọi tập con** của chỉ số để giữ lại: với mỗi `mask`, dựng dãy con; nếu nó là mảng sắc (mọi cặp kề có XOR khác 0), cập nhật số phần tử giữ được lớn nhất. Số xóa ít nhất là `n − (số phần tử giữ lớn nhất)`. Cách này chỉ dùng được với `n` nhỏ nhưng chắc chắn đúng theo đề.

In [9]:
def min_deletions_subsets(nums: list[int]) -> int:
    n = len(nums)
    best_kept = 0
    for mask in range(1 << n):                                   # mỗi mask là một cách giữ
        kept = [nums[i] for i in range(n) if (mask >> i) & 1]
        if all(a ^ b != 0 for a, b in zip(kept, kept[1:])):      # mảng sắc?
            best_kept = max(best_kept, len(kept))
    return n - best_kept


for nums, expected in examples:
    print(f"nums={nums} -> {min_deletions_subsets(nums)} (kỳ vọng {expected})")

nums=[1, 1, 2, 2, 2, 3] -> 3 (kỳ vọng 3)
nums=[4, 4, 9, 4] -> 1 (kỳ vọng 1)


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Có `2ⁿ` tập con, mỗi tập kiểm tra `O(n)`: **Time `O(2ⁿ · n)`**, **Space `O(n)`**. Với `n = 100` là hoàn toàn không khả thi; dùng được tối đa `n ≈ 16–18`.

**Nút thắt.** Số cách chọn bùng nổ theo cấp số nhân. Cần một cách xây dựng đáp án mà không thử mọi tập con: ta sẽ dùng quy hoạch động (tăng dần), rồi nhận ra một cấu trúc rất đơn giản.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát then chốt.** Điều kiện sắc chỉ cấm hai phần tử **kề nhau** bằng nhau. Nếu hai phần tử **cùng một đoạn** (liên tiếp, cùng giá trị) đều được giữ thì mọi phần tử nằm giữa chúng cũng thuộc đoạn đó; trong dãy con còn lại, hai phần tử giữ **gần nhau nhất** trong đoạn sẽ kề nhau và bằng nhau, vi phạm. Vậy **mỗi đoạn giữ tối đa một phần tử**.

Ngược lại, giữ **đúng một phần tử mỗi đoạn** là hợp lệ: hai đoạn kề nhau có giá trị khác nhau nên các phần tử giữ lại kề nhau đều khác nhau.

Do đó số phần tử giữ được lớn nhất là **số đoạn**, và

```
số lần xóa nhỏ nhất = n − (số đoạn) = số cặp kề bằng nhau
```

(mỗi đoạn dài `k` đóng góp `k − 1` cặp kề bằng nhau và cần xóa `k − 1` phần tử). Chỉ cần **một lượt duyệt** đếm các vị trí `nums[i] == nums[i − 1]`.

In [10]:
nums = [1, 1, 2, 2, 2, 3]
n = len(nums)
runs = [len(list(g)) for _, g in groupby(nums)]
print("độ dài các đoạn:", runs)
print("giữ tối đa một phần tử mỗi đoạn -> giữ được", len(runs), "phần tử")
print("xóa tối thiểu = n - số đoạn =", n - len(runs), "= sum(k - 1) =", sum(k - 1 for k in runs))

độ dài các đoạn: [2, 3, 1]
giữ tối đa một phần tử mỗi đoạn -> giữ được 3 phần tử
xóa tối thiểu = n - số đoạn = 3 = sum(k - 1) = 3


## 7. Các cách tiếp theo

### Cách 2 — Quy hoạch động `O(n²)`: dãy con sắc dài nhất

**Giải thích.** Gọi `dp[i]` là độ dài **lớn nhất** của một dãy con sắc **kết thúc ở phần tử `i`** (phần tử `i` được giữ và là phần tử cuối). Phần tử đứng ngay trước nó trong dãy con là `nums[j]` với `j < i` và `nums[j] != nums[i]`, nên

```
dp[i] = 1 + max(dp[j] for j < i if nums[j] != nums[i])      (hoặc 1 nếu không có j nào)
```

Số phần tử giữ lớn nhất là `max(dp)`, đáp án `n − max(dp)`. Cách này đúng cả khi điều kiện phức tạp hơn "khác nhau", nhưng chậm hơn.

In [11]:
def min_deletions_dp(nums: list[int]) -> int:
    n = len(nums)
    dp = [1] * n                                   # dp[i]: dãy con sắc dài nhất kết thúc tại i
    for i in range(n):
        for j in range(i):
            if nums[j] ^ nums[i] != 0:             # nums[j] đứng ngay trước nums[i] trong dãy con
                dp[i] = max(dp[i], dp[j] + 1)
    return n - max(dp)


for nums, expected in examples:
    print(f"nums={nums} -> {min_deletions_dp(nums)} (kỳ vọng {expected})")

nums=[1, 1, 2, 2, 2, 3] -> 3 (kỳ vọng 3)
nums=[4, 4, 9, 4] -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 2:** **Time `O(n²)`** (hai vòng lặp lồng nhau), **Space `O(n)`** cho `dp`. Với `n ≤ 100` chạy tức thì.

### Cách 3 — Tham lam bằng ngăn xếp: gặp trùng với phần tử vừa giữ thì xóa

**Giải thích.** Duyệt từ trái sang phải, giữ list `kept` các phần tử đã giữ và biến `deleted`. Với mỗi `x`: nếu `kept` không rỗng và `kept[-1] == x` thì **xóa** `x` (`deleted += 1`), ngược lại **giữ** `x` (`kept.append(x)`). Ý tưởng: luôn giữ phần tử **đầu tiên** của mỗi đoạn và xóa các phần tử còn lại của đoạn.

In [12]:
def min_deletions_stack(nums: list[int]) -> int:
    kept = []
    deleted = 0
    for x in nums:
        if kept and kept[-1] == x:      # trùng phần tử vừa giữ: xóa x
            deleted += 1
        else:
            kept.append(x)              # khác phần tử vừa giữ: giữ
    return deleted


for nums, expected in examples:
    print(f"nums={nums} -> {min_deletions_stack(nums)} (kỳ vọng {expected})")

nums=[1, 1, 2, 2, 2, 3] -> 3 (kỳ vọng 3)
nums=[4, 4, 9, 4] -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(n)`** cho `kept` (có thể thay bằng một biến `prev` để còn `O(1)`).

### Cách 4 — Đếm số cặp kề bằng nhau (cách của lời giải tham khảo)

**Giải thích.** Theo bước đột phá, đáp án là **số vị trí `i` với `nums[i] == nums[i − 1]`**. Dùng `pairwise` để duyệt các cặp kề và cộng `True` (`= 1`). Một dòng, không cần danh sách phụ.

In [13]:
def min_deletions_pairs(nums: list[int]) -> int:
    return sum(x == y for x, y in pairwise(nums))


for nums, expected in examples:
    print(f"nums={nums} -> {min_deletions_pairs(nums)} (kỳ vọng {expected})")

nums=[1, 1, 2, 2, 2, 3] -> 3 (kỳ vọng 3)
nums=[4, 4, 9, 4] -> 1 (kỳ vọng 1)


In [14]:
# Code nộp LeetCode cho Cách 4
from itertools import pairwise
from typing import List


class Solution:
    def minDeletions(self, nums: List[int]) -> int:
        return sum(x == y for x, y in pairwise(nums))

**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`** (`pairwise` là bộ sinh, không tạo list).

### Cách 5 — Đếm số đoạn bằng `groupby`

**Giải thích.** Số phần tử giữ lớn nhất bằng số đoạn bằng nhau; `groupby(nums)` tách chính xác các đoạn. Đáp án `n − len(list(groupby(nums)))`.

In [15]:
def min_deletions_groupby(nums: list[int]) -> int:
    runs = sum(1 for _ in groupby(nums))          # số đoạn bằng nhau liên tiếp
    return len(nums) - runs


for nums, expected in examples:
    print(f"nums={nums} -> {min_deletions_groupby(nums)} (kỳ vọng {expected})")

nums=[1, 1, 2, 2, 2, 3] -> 3 (kỳ vọng 3)
nums=[4, 4, 9, 4] -> 1 (kỳ vọng 1)


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

> Cách 4 là cách nộp bài gọn nhất. Cách 1 và 2 là chuẩn đối chiếu độc lập (không dựa vào quan sát). Cách 3 cho thấy lập luận "xóa khi trùng phần tử vừa giữ".

## 8. TRACE từng bước

Ta theo dõi Cách 3 (ngăn xếp) trên `nums = [1, 1, 2, 2, 2, 3]`, Cách 4 (`pairwise` và XOR) trên cùng input, và Cách 2 (DP) trên `[4, 4, 9, 4]`.

In [16]:
def trace_stack(nums):
    print(f"nums={nums}")
    kept, deleted = [], 0
    for i, x in enumerate(nums):
        before = list(kept)
        if kept and kept[-1] == x:
            deleted += 1
            action = f"trùng {kept[-1]} -> XÓA (deleted={deleted})"
        else:
            kept.append(x)
            action = "giữ"
        print(f"  i={i}, x={x}, kept trước={before}: {action}; kept sau={kept}")
    print("  => số lần xóa:", deleted, "| mảng còn lại:", kept)
    return deleted


assert trace_stack([1, 1, 2, 2, 2, 3]) == 3

nums=[1, 1, 2, 2, 2, 3]
  i=0, x=1, kept trước=[]: giữ; kept sau=[1]
  i=1, x=1, kept trước=[1]: trùng 1 -> XÓA (deleted=1); kept sau=[1]
  i=2, x=2, kept trước=[1]: giữ; kept sau=[1, 2]
  i=3, x=2, kept trước=[1, 2]: trùng 2 -> XÓA (deleted=2); kept sau=[1, 2]
  i=4, x=2, kept trước=[1, 2]: trùng 2 -> XÓA (deleted=3); kept sau=[1, 2]
  i=5, x=3, kept trước=[1, 2]: giữ; kept sau=[1, 2, 3]
  => số lần xóa: 3 | mảng còn lại: [1, 2, 3]


In [17]:
def trace_pairs(nums):
    print(f"nums={nums}")
    total = 0
    for i, (x, y) in enumerate(pairwise(nums)):
        xor = x ^ y
        total += xor == 0
        print(f"  cặp (i={i}, {i + 1}): {x} ^ {y} = {xor} -> {'bằng nhau, đếm +1' if xor == 0 else 'khác nhau'} (tổng {total})")
    print("  => đáp án:", total)
    return total


assert trace_pairs([1, 1, 2, 2, 2, 3]) == 3
print()
assert trace_pairs([4, 4, 9, 4]) == 1


def trace_dp(nums):
    n = len(nums)
    dp = [1] * n
    print(f"nums={nums}")
    for i in range(n):
        for j in range(i):
            if nums[j] ^ nums[i] != 0:
                if dp[j] + 1 > dp[i]:
                    dp[i] = dp[j] + 1
        print(f"  dp[{i}] ({nums[i]}) = {dp[i]}")
    print(f"  max(dp) = {max(dp)} -> xóa {n} - {max(dp)} = {n - max(dp)}")
    return n - max(dp)


assert trace_dp([4, 4, 9, 4]) == 1

nums=[1, 1, 2, 2, 2, 3]
  cặp (i=0, 1): 1 ^ 1 = 0 -> bằng nhau, đếm +1 (tổng 1)
  cặp (i=1, 2): 1 ^ 2 = 3 -> khác nhau (tổng 1)
  cặp (i=2, 3): 2 ^ 2 = 0 -> bằng nhau, đếm +1 (tổng 2)
  cặp (i=3, 4): 2 ^ 2 = 0 -> bằng nhau, đếm +1 (tổng 3)
  cặp (i=4, 5): 2 ^ 3 = 1 -> khác nhau (tổng 3)
  => đáp án: 3

nums=[4, 4, 9, 4]
  cặp (i=0, 1): 4 ^ 4 = 0 -> bằng nhau, đếm +1 (tổng 1)
  cặp (i=1, 2): 4 ^ 9 = 13 -> khác nhau (tổng 1)
  cặp (i=2, 3): 9 ^ 4 = 13 -> khác nhau (tổng 1)
  => đáp án: 1
nums=[4, 4, 9, 4]
  dp[0] (4) = 1
  dp[1] (4) = 1
  dp[2] (9) = 2
  dp[3] (4) = 3
  max(dp) = 3 -> xóa 4 - 3 = 1


Bảng tóm tắt (Cách 3, `nums = [1,1,2,2,2,3]`):

| `i` | `x` | `kept` trước | hành động | `deleted` |
|---|---|---|---|---|
| 0 | `1` | `[]` | giữ | 0 |
| 1 | `1` | `[1]` | trùng, **xóa** | 1 |
| 2 | `2` | `[1]` | giữ | 1 |
| 3 | `2` | `[1, 2]` | trùng, **xóa** | 2 |
| 4 | `2` | `[1, 2]` | trùng, **xóa** | 3 |
| 5 | `3` | `[1, 2]` | giữ | 3 |

Mảng còn lại `[1, 2, 3]`, đáp án `3`. Với `[4,4,9,4]` DP cho `dp = [1, 1, 2, 3]`: dãy con sắc dài nhất `[4, 9, 4]` độ dài `3`, nên xóa `4 − 3 = 1`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (đếm số cặp kề bằng nhau bằng `pairwise`). Đúng class `Solution`, đúng method `minDeletions(self, nums: List[int]) -> int`.

In [18]:
from itertools import pairwise
from typing import List


class Solution:
    def minDeletions(self, nums: List[int]) -> int:
        # a ^ b == 0  <=>  a == b: mỗi cặp kề bằng nhau cần xóa đúng một phần tử
        return sum(x == y for x, y in pairwise(nums))


for nums, expected in examples:
    assert Solution().minDeletions(nums) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Điều kiện sắc là "kề nhau thì khác nhau":** `a ^ b = 0 ⇔ a = b` (mục 3.2). Nên mảng sắc là mảng không có hai phần tử kề nhau bằng nhau.
- **Cận trên (xóa ít nhất bấy nhiêu):** chia `nums` thành các đoạn bằng nhau liên tiếp tối đại. Trong một dãy con sắc, mỗi đoạn có **nhiều nhất một** phần tử được giữ: nếu giữ hai phần tử `p < q` cùng đoạn thì mọi phần tử giữa chúng cũng cùng giá trị (cùng đoạn); trong dãy con, hai phần tử giữ **gần nhau nhất** của đoạn đó đứng kề nhau (không còn phần tử khác giữa chúng) và bằng nhau, vi phạm. Vậy số phần tử giữ được `≤` số đoạn, nên số xóa `≥ n −` số đoạn.
- **Đạt được cận:** giữ đúng một phần tử mỗi đoạn. Hai đoạn kề nhau có giá trị khác nhau (nếu bằng nhau, chúng đã là một đoạn), nên các phần tử được giữ kề nhau đều khác nhau: mảng sắc. Số xóa `= n −` số đoạn.
- **Đếm bằng cặp kề:** đoạn dài `k` chứa đúng `k − 1` cặp kề bằng nhau; các cặp kề thuộc hai đoạn khác nhau thì khác giá trị. Nên số cặp kề bằng nhau `= Σ(k − 1) = n −` số đoạn.
- **Không được xóa nguyên một đoạn:** xóa hết một đoạn làm hai đoạn hai bên trở nên kề nhau và có thể bằng nhau (`[4,4,9,4]` bỏ `9` còn `[4,4,4]`). Đáp án tối ưu không cần làm điều đó.
- **Chuẩn đối chiếu:** Cách 1 (mọi tập con) và Cách 2 (DP) không dùng quan sát nào, được so sánh với Cách 4 trong phần test.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- **Cách 1 (mọi tập con):** Time `O(2ⁿ · n)`, Space `O(n)`; chỉ dùng cho `n` nhỏ.
- **Cách 2 (DP `O(n²)`):** Time `O(n²)`, Space `O(n)`.
- **Cách 3 (ngăn xếp tham lam):** Time `O(n)`, Space `O(n)` (hoặc `O(1)` với biến `prev`).
- **Cách 4 (đếm cặp kề bằng nhau):** Time `O(n)`, Space `O(1)`.
- **Cách 5 (`groupby`):** Time `O(n)`, Space `O(1)`.
- Trường hợp trung bình và xấu nhất như nhau (luôn duyệt hết mảng). Không thể nhanh hơn `O(n)` vì phải đọc hết dữ liệu.

## 12. Bộ test

Ta gom các cách vào từ điển `impls`. Hai lời giải **độc lập** làm chuẩn: `oracle_subsets` (Cách 1, mảng ngắn `n ≤ 12`) và `oracle_dp` (Cách 2, mảng dài). Cả hai không dùng quan sát "số cặp kề bằng nhau". Ngoài ra tập `impls` có cả các cách tối ưu (3, 4, 5) và bản nộp.

In [19]:
sol = Solution()
impls_fast = {
    "dp": min_deletions_dp,
    "stack": min_deletions_stack,
    "pairs": min_deletions_pairs,
    "groupby": min_deletions_groupby,
    "submit": sol.minDeletions,
}
impls_all = dict(impls_fast, subsets=min_deletions_subsets)
oracle_subsets = min_deletions_subsets
oracle_dp = min_deletions_dp

for nums, expected in examples:
    for name, fn in impls_all.items():
        assert fn(list(nums)) == expected, (name, nums, fn(list(nums)), expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [20]:
def check_all(nums, expected, label):
    for name, fn in (impls_all if len(nums) <= 14 else impls_fast).items():
        original = list(nums)
        got = fn(original)
        assert got == expected, (label, name, got, expected)
        assert original == nums, (label, name, "đã sửa đầu vào")
    assert oracle_dp(nums) == expected
    print("PASS", label, "->", expected)


check_all([5, 5], 1, "n = 2, hai phần tử bằng nhau")
check_all([5, 6], 0, "n = 2, khác nhau")
check_all([1, 2, 3, 4], 0, "mọi phần tử khác nhau, đã là mảng sắc")
check_all([7, 7, 7, 7], 3, "toàn một giá trị: giữ một, xóa n - 1")
check_all([1, 2, 1, 2, 1], 0, "xen kẽ, các giá trị trùng nhưng không kề nhau")
check_all([1, 2, 2, 1], 1, "một cặp bằng nhau ở giữa")
check_all([1, 1, 1, 2, 2, 1, 1], 4, "ba đoạn 3, 2, 2: xóa 2 + 1 + 1")
check_all([4, 4, 9, 4], 1, "ví dụ 2 của đề: không xóa 9")
check_all([3, 1, 1, 3], 1, "hai đầu bằng nhau nhưng không kề nhau")
check_all([100, 100, 1, 1], 2, "biên giá trị 100 và 1")
check_all([9] * 100, 99, "n = 100, toàn một giá trị")
check_all([1, 2] * 50, 0, "n = 100, xen kẽ hoàn toàn")
check_all(list(range(1, 101)), 0, "n = 100, mọi giá trị khác nhau")

PASS n = 2, hai phần tử bằng nhau -> 1
PASS n = 2, khác nhau -> 0
PASS mọi phần tử khác nhau, đã là mảng sắc -> 0
PASS toàn một giá trị: giữ một, xóa n - 1 -> 3
PASS xen kẽ, các giá trị trùng nhưng không kề nhau -> 0
PASS một cặp bằng nhau ở giữa -> 1
PASS ba đoạn 3, 2, 2: xóa 2 + 1 + 1 -> 4
PASS ví dụ 2 của đề: không xóa 9 -> 1
PASS hai đầu bằng nhau nhưng không kề nhau -> 1
PASS biên giá trị 100 và 1 -> 2
PASS n = 100, toàn một giá trị -> 99
PASS n = 100, xen kẽ hoàn toàn -> 0
PASS n = 100, mọi giá trị khác nhau -> 0


### 12.2. Tính chất và phản ví dụ

In [21]:
random.seed(4083)

# Tính chất 1: 0 <= đáp án <= n - 1; đáp án = n - số đoạn
for _ in range(500):
    n = random.randint(2, 30)
    nums = [random.randint(1, 3) for _ in range(n)]
    a = sol.minDeletions(nums)
    runs = sum(1 for _ in groupby(nums))
    assert 0 <= a <= n - 1 and a == n - runs
print("PASS tính chất: 0 <= đáp án <= n - 1 và đáp án = n - số đoạn")

# Tính chất 2: đổi tên giá trị (song ánh) hoặc đảo ngược mảng không đổi đáp án
for _ in range(300):
    n = random.randint(2, 20)
    nums = [random.randint(1, 4) for _ in range(n)]
    mapping = dict(zip(range(1, 5), random.sample(range(1, 101), 4)))
    renamed = [mapping[x] for x in nums]
    assert sol.minDeletions(nums) == sol.minDeletions(renamed) == sol.minDeletions(nums[::-1])
print("PASS tính chất: đổi tên giá trị / đảo ngược không đổi đáp án")

# Tính chất 3: nhân đôi mỗi phần tử (x -> x, x) cộng thêm đúng n lần xóa
for _ in range(300):
    n = random.randint(2, 20)
    nums = [random.randint(1, 4) for _ in range(n)]
    doubled = [x for x in nums for _ in range(2)]
    assert sol.minDeletions(doubled) == sol.minDeletions(nums) + n
print("PASS tính chất: nhân đôi mỗi phần tử cộng thêm đúng n lần xóa")

# Phản ví dụ: đếm số giá trị trùng (không cần kề nhau) làm đáp án sai
def sai_dem_trung(nums):
    return len(nums) - len(set(nums))

print("[4,4,9,4] -> sai:", sai_dem_trung([4, 4, 9, 4]), "| đúng:", sol.minDeletions([4, 4, 9, 4]))
assert sai_dem_trung([4, 4, 9, 4]) == 2 and sol.minDeletions([4, 4, 9, 4]) == 1

PASS tính chất: 0 <= đáp án <= n - 1 và đáp án = n - số đoạn
PASS tính chất: đổi tên giá trị / đảo ngược không đổi đáp án
PASS tính chất: nhân đôi mỗi phần tử cộng thêm đúng n lần xóa
[4,4,9,4] -> sai: 2 | đúng: 1


### 12.3. Đối chiếu ngẫu nhiên (có seed) với hai oracle độc lập

In [22]:
random.seed(2026)
for trial in range(1500):                          # mảng ngắn: so với oracle_subsets (không dùng quan sát nào)
    n = random.randint(2, 12)
    hi = random.choice([2, 3, 5, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    expected = oracle_subsets(nums)
    for name, fn in impls_all.items():
        assert fn(list(nums)) == expected, (name, nums, fn(list(nums)), expected)

for trial in range(1500):                          # mảng dài tới 100: so với oracle_dp
    n = random.randint(2, 100)
    hi = random.choice([2, 3, 5, 100])
    nums = [random.randint(1, hi) for _ in range(n)]
    expected = oracle_dp(nums)
    for name, fn in impls_fast.items():
        assert fn(list(nums)) == expected, (name, nums)
print("PASS 3000 input ngẫu nhiên: mọi cách khớp hai oracle độc lập (mọi tập con và quy hoạch động)")

PASS 3000 input ngẫu nhiên: mọi cách khớp hai oracle độc lập (mọi tập con và quy hoạch động)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Xóa cả hai phần tử của mỗi cặp bằng nhau

In [23]:
def sai_xoa_ca_hai(nums):
    deleted = 0
    i = 0
    while i < len(nums) - 1:
        if nums[i] == nums[i + 1]:
            deleted += 2                     # xóa cả hai
            i += 2
        else:
            i += 1
    return deleted


print("[1,1,2,2,2,3] -> sai:", sai_xoa_ca_hai([1, 1, 2, 2, 2, 3]), "| đúng:", Solution().minDeletions([1, 1, 2, 2, 2, 3]))
print("[4,4,9,4]     -> sai:", sai_xoa_ca_hai([4, 4, 9, 4]), "| đúng:", Solution().minDeletions([4, 4, 9, 4]))

[1,1,2,2,2,3] -> sai: 4 | đúng: 3
[4,4,9,4]     -> sai: 2 | đúng: 1


**Vì sao sai:** để hai phần tử bằng nhau không còn kề nhau, chỉ cần xóa **một** trong hai. Xóa cả hai gấp đôi chi phí (ví dụ 2 cho `2` thay vì `1`). Trong một đoạn dài `k` cần xóa `k − 1`, không phải `k`.

### Lỗi 2: Đếm số giá trị trùng lặp (không cần kề nhau)

In [24]:
def sai_dem_trung_lap(nums):
    return len(nums) - len(set(nums))


print("[4,4,9,4] -> sai:", sai_dem_trung_lap([4, 4, 9, 4]), "| đúng:", Solution().minDeletions([4, 4, 9, 4]))
print("[1,2,1,2,1] -> sai:", sai_dem_trung_lap([1, 2, 1, 2, 1]), "| đúng:", Solution().minDeletions([1, 2, 1, 2, 1]))

[4,4,9,4] -> sai: 2 | đúng: 1
[1,2,1,2,1] -> sai: 3 | đúng: 0


**Vì sao sai:** điều kiện chỉ cấm hai phần tử **kề nhau** bằng nhau. `[1, 2, 1, 2, 1]` có giá trị lặp lại nhưng không có cặp kề nào bằng nhau, nên đã sắc (`0` lần xóa). `n − len(set(nums))` đếm mọi bản sao dư thừa bất kể vị trí nên cho `3`.

### Lỗi 3: So sánh vòng tròn (`nums[-1]`) khi `i = 0`

In [25]:
def sai_vong_tron(nums):
    count = 0
    for i in range(len(nums)):               # sai: i = 0 so nums[0] với nums[-1] (phần tử cuối)
        if nums[i] == nums[i - 1]:
            count += 1
    return count


print("[4,4,9,4] -> sai:", sai_vong_tron([4, 4, 9, 4]), "| đúng:", Solution().minDeletions([4, 4, 9, 4]))
print("[3,1,1,3] -> sai:", sai_vong_tron([3, 1, 1, 3]), "| đúng:", Solution().minDeletions([3, 1, 1, 3]))

[4,4,9,4] -> sai: 2 | đúng: 1
[3,1,1,3] -> sai: 2 | đúng: 1


**Vì sao sai:** trong Python `nums[-1]` là phần tử **cuối**, nên ở `i = 0` ta so sánh phần tử đầu với phần tử cuối, thêm một "cặp kề" không tồn tại (mảng không quay vòng). Phải duyệt `i` từ `1` đến `n − 1` (so `nums[i]` với `nums[i − 1]`), hoặc dùng `pairwise` / `zip(nums, nums[1:])`.

### Lỗi 4: Hiểu ngược điều kiện XOR

In [26]:
def sai_dao_dieu_kien(nums):
    return sum(1 for x, y in pairwise(nums) if x ^ y)       # x ^ y truthy khi KHÁC nhau


print("[1,1,2,2,2,3] -> sai:", sai_dao_dieu_kien([1, 1, 2, 2, 2, 3]), "| đúng:", Solution().minDeletions([1, 1, 2, 2, 2, 3]))

[1,1,2,2,2,3] -> sai: 2 | đúng: 3


**Vì sao sai:** trong Python `x ^ y` là số nguyên; nó **truthy** (khác 0) khi `x ≠ y` và `0` (falsy) khi `x == y`. Viết `if x ^ y` đếm các cặp **khác nhau**. Cặp cần xóa là cặp có `x ^ y == 0`, tức `x == y`. Luôn đọc kỹ: "XOR **khác** 0" là điều kiện tốt; "XOR **bằng** 0" là điều kiện xấu cần xóa.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**XOR bằng 0**" của hai số: đó là **hai số bằng nhau**. Thay điều kiện bit bằng điều kiện bằng nhau để đơn giản bài toán.
- "Xóa ít nhất để **không còn hai phần tử kề nhau bằng nhau**" → mỗi **đoạn các số bằng nhau** giữ lại **một** phần tử → đáp án = **số cặp kề bằng nhau** = `n − số đoạn`. Bài cùng họ: LeetCode 1578 (Minimum Time to Make Rope Colorful), 1047 (Remove All Adjacent Duplicates).
- "Giữ thứ tự và chỉ xóa" nghĩa là chọn một **dãy con**; khi muốn chắc chắn, thử **brute force bitmask** (`n ≤ 16`) hoặc **DP `O(n²)`** làm chuẩn đối chiếu.
- Không xóa nguyên một đoạn: có thể làm hai đoạn hai bên chạm nhau và trùng giá trị.
- Duyệt cặp kề với `pairwise(nums)` hoặc `zip(nums, nums[1:])`, tránh `nums[i - 1]` khi `i = 0`.
- Nếu mỗi phần tử có **chi phí xóa** riêng, mỗi đoạn giữ phần tử **đắt nhất** (xem biến thể).

## 15. Biến thể của bài

1. **Trả về mảng sắc kết quả** (không chỉ số lần xóa).
2. **Có chi phí xóa riêng** cho mỗi phần tử; tối thiểu hóa tổng chi phí (mỗi đoạn giữ phần tử có chi phí lớn nhất — LeetCode 1578).
3. **Chỉ được xóa phần tử liên tiếp (một đoạn)** hoặc giới hạn số lần xóa: cần xem xét thêm.
4. **Số đoạn bằng nhau** (đếm `groupby`) hoặc **độ dài đoạn dài nhất**.
5. **Điều kiện khác thay cho XOR khác 0**: ví dụ tổng của hai phần tử kề nhau **khác chẵn** (cần DP).

In [27]:
def make_sharp(nums):
    """Biến thể 1: mảng sắc thu được (giữ phần tử đầu mỗi đoạn)."""
    return [value for value, _ in groupby(nums)]


def min_cost_sharp(nums, cost):
    """Biến thể 2: chi phí xóa riêng; mỗi đoạn giữ phần tử có chi phí lớn nhất."""
    total = 0
    i = 0
    n = len(nums)
    while i < n:
        j = i
        group_sum, group_max = 0, 0
        while j < n and nums[j] == nums[i]:
            group_sum += cost[j]
            group_max = max(group_max, cost[j])
            j += 1
        total += group_sum - group_max
        i = j
    return total


def longest_run(nums):
    """Biến thể 4: độ dài đoạn bằng nhau dài nhất."""
    return max(len(list(g)) for _, g in groupby(nums))


def min_cost_bruteforce(nums, cost):
    """Chuẩn đối chiếu cho biến thể 2 (mọi tập con)."""
    n = len(nums)
    best = None
    for mask in range(1 << n):
        kept = [nums[i] for i in range(n) if (mask >> i) & 1]
        if all(a != b for a, b in zip(kept, kept[1:])):
            c = sum(cost[i] for i in range(n) if not (mask >> i) & 1)
            best = c if best is None else min(best, c)
    return best


assert make_sharp([1, 1, 2, 2, 2, 3]) == [1, 2, 3]
assert make_sharp([4, 4, 9, 4]) == [4, 9, 4]
assert longest_run([1, 1, 2, 2, 2, 3]) == 3
random.seed(15)
for _ in range(300):
    n = random.randint(2, 10)
    nums = [random.randint(1, 3) for _ in range(n)]
    cost = [random.randint(1, 9) for _ in range(n)]
    assert min_cost_sharp(nums, cost) == min_cost_bruteforce(nums, cost)
    assert len(make_sharp(nums)) == n - sol.minDeletions(nums)
assert min_cost_sharp([1, 1, 2, 2, 2, 3], [1, 1, 1, 1, 1, 1]) == 3
print("PASS biến thể")

PASS biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tập con | bitmask, giữ nhiều phần tử nhất mà sắc | `O(2ⁿ·n)` | `O(n)` | chuẩn đối chiếu `n ≤ 16` |
| 2. DP `O(n²)` | `dp[i] = 1 + max dp[j]` với `nums[j] != nums[i]` | `O(n²)` | `O(n)` | chuẩn đối chiếu, điều kiện tổng quát hơn |
| 3. Ngăn xếp tham lam | xóa khi trùng phần tử vừa giữ | `O(n)` | `O(n)` / `O(1)` | thấy cơ chế "giữ phần tử đầu mỗi đoạn" |
| 4. Đếm cặp kề bằng nhau | `sum(x == y for x, y in pairwise(nums))` | `O(n)` | `O(1)` | **cách nộp bài** |
| 5. `n − số đoạn` | `groupby` | `O(n)` | `O(1)` | khi cần cả danh sách đoạn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `5 ^ 5`, `3 ^ 5`, `list(pairwise([1, 1, 2]))`, `Solution().minDeletions([2, 2, 2])`, `Solution().minDeletions([1, 2, 1, 2])`.

**Bài 2 (tính tay).** Với `nums = [3, 3, 3, 5, 5, 3, 3]`: liệt kê các đoạn bằng nhau, số lần xóa tối thiểu và mảng sắc thu được. Làm lại với `nums = [6, 7, 7, 6, 6, 7]`.

**Bài 3 (điền TODO).** Hoàn thành `sharp_todo(nums)` theo Cách 3 (ngăn xếp).

In [28]:
def sharp_todo(nums):
    kept = []
    deleted = 0
    for x in nums:
        # TODO 1: nếu kept không rỗng và kept[-1] == x thì tăng deleted
        # TODO 2: ngược lại thì kept.append(x)
        pass
    return deleted

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `sharp_groupby2(nums)` theo Cách 5 (`n − số đoạn`) mà không nhìn code mẫu, rồi tự viết thêm một bản **không dùng `itertools`** (chỉ vòng `for` và so sánh `nums[i]` với `nums[i - 1]` từ `i = 1`). Kiểm tra bằng hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `sharp_result(nums)` trả về **mảng sắc** thu được sau khi xóa tối thiểu (giữ phần tử đầu mỗi đoạn). (b) Viết `sharp_min_cost(nums, cost)` trả về **tổng chi phí xóa nhỏ nhất** khi xóa phần tử `i` tốn `cost[i]` (mỗi đoạn giữ phần tử đắt nhất).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `a ^ a = 0`; `pairwise` cho các cặp kề; mỗi cặp kề bằng nhau cần xóa một. Bài 2: chia mảng thành các đoạn các số bằng nhau liên tiếp.
- **Mức 2 (vừa):** Bài 1: `0`, `6`, `[(1, 1), (1, 2)]`, `2`, `0`. Bài 2: `[3,3,3,5,5,3,3]` có các đoạn `[3,3,3]`, `[5,5]`, `[3,3]` nên xóa `2 + 1 + 1 = 4`, mảng sắc `[3, 5, 3]`. `[6,7,7,6,6,7]`: đoạn `[6]`, `[7,7]`, `[6,6]`, `[7]`: xóa `0 + 1 + 1 + 0 = 2`, mảng `[6, 7, 6, 7]`. Bài 3: hai TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `n − sum(1 for _ in groupby(nums))`; bản không `itertools`: `sum(1 for i in range(1, len(nums)) if nums[i] == nums[i - 1])`. Bài 5(a): `[v for v, _ in groupby(nums)]`. Bài 5(b): với mỗi đoạn cộng `tổng chi phí − chi phí lớn nhất`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert 5 ^ 5 == 0 and 3 ^ 5 == 6
assert list(pairwise([1, 1, 2])) == [(1, 1), (1, 2)]
assert Solution().minDeletions([2, 2, 2]) == 2
assert Solution().minDeletions([1, 2, 1, 2]) == 0

# Bài 2
assert [len(list(g)) for _, g in groupby([3, 3, 3, 5, 5, 3, 3])] == [3, 2, 2]
assert Solution().minDeletions([3, 3, 3, 5, 5, 3, 3]) == 4
assert make_sharp([3, 3, 3, 5, 5, 3, 3]) == [3, 5, 3]
assert Solution().minDeletions([6, 7, 7, 6, 6, 7]) == 2
assert make_sharp([6, 7, 7, 6, 6, 7]) == [6, 7, 6, 7]

# Bài 3
def sharp_todo_ans(nums):
    kept = []
    deleted = 0
    for x in nums:
        if kept and kept[-1] == x:
            deleted += 1
        else:
            kept.append(x)
    return deleted

for nums, expected in examples:
    assert sharp_todo_ans(nums) == expected

# Bài 4
def sharp_groupby2(nums):
    return len(nums) - sum(1 for _ in groupby(nums))

def sharp_plain(nums):
    return sum(1 for i in range(1, len(nums)) if nums[i] == nums[i - 1])

for nums, expected in examples:
    assert sharp_groupby2(nums) == expected and sharp_plain(nums) == expected

# Bài 5(a), 5(b)
def sharp_result(nums):
    return [v for v, _ in groupby(nums)]

def sharp_min_cost(nums, cost):
    total = 0
    i = 0
    while i < len(nums):
        j = i
        while j < len(nums) and nums[j] == nums[i]:
            j += 1
        total += sum(cost[i:j]) - max(cost[i:j])
        i = j
    return total

assert sharp_result([1, 1, 2, 2, 2, 3]) == [1, 2, 3]
assert sharp_min_cost([1, 1, 2, 2, 2, 3], [5, 1, 4, 2, 3, 7]) == 1 + (4 + 2 + 3 - 4)
assert sharp_min_cost([4, 4, 9, 4], [3, 8, 1, 2]) == 3
random.seed(19)
for _ in range(200):
    n = random.randint(2, 9)
    nums = [random.randint(1, 3) for _ in range(n)]
    cost = [random.randint(1, 9) for _ in range(n)]
    assert sharp_min_cost(nums, cost) == min_cost_bruteforce(nums, cost)
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm số cặp kề bằng nhau)

```python
from itertools import pairwise
from typing import List


class Solution:
    def minDeletions(self, nums: List[int]) -> int:
        return sum(x == y for x, y in pairwise(nums))
```

**Checklist**

- [ ] `a ^ b == 0` nghĩa là `a == b`; mảng sắc = **không có hai phần tử kề nhau bằng nhau**?
- [ ] Mỗi **đoạn** các số bằng nhau giữ **một** phần tử: số xóa = `k − 1` mỗi đoạn, tổng = số cặp kề bằng nhau?
- [ ] Chỉ so sánh các cặp **kề nhau** (không so sánh vòng tròn, không đếm giá trị trùng ở xa)?
- [ ] Không xóa **cả hai** phần tử của cặp bằng nhau?
- [ ] Duyệt cặp bằng `pairwise` / `zip(nums, nums[1:])` hoặc `i` từ `1`?
- [ ] Đã thử: hai phần tử bằng nhau, toàn một giá trị, xen kẽ, hai đầu bằng nhau nhưng không kề?
- [ ] Đã đối chiếu với brute force (mọi tập con) hoặc DP trên input ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def minDeletions(self, nums: List[int]) -> int`, có `import` cần thiết?